# Módulo AI3 · Soluciones

## Setup (pipeline del notebook: chunking, RAG, tools, router)

In [ ]:
import textwrap

SYSTEM_RAG = textwrap.dedent("""
Eres el asistente de pedidos de TiendaSur.

Responde SOLO con la información de los fragmentos entre <docs>.
- Cita el fragmento con [id del chunk].
- Si el fragmento no contesta la pregunta, di exactamente: "no lo sé".
""")

AGENTE_SYSTEM = textwrap.dedent(SYSTEM_RAG + """

Además tienes herramientas útiles para resolver incidencias:
- `buscar_docs(consulta)`: busca en la guía.
- `get_pedido(pedido_id)`: estado de un pedido.
- `crear_ticket(tema, pedido_id)`: registra la incidencia y devuelve su id.

Plan: consulta las docs o el pedido que necesites, y para problemas reales
crea el ticket y ciérrate respondiendo el id.""")

print('setup + prompts ok; [API] las celdas del agente se ejecutan con clave')


## E1 · Chunking con solape (2 chunks por sección, 15 % solape)

In [ ]:
import numpy as np

import re

GUIA = """## Guía de pedidos de TiendaSur

### Envíos
El envío estándar tarda 3-5 días laborables. Los pedidos urgentes llegan al día siguiente si se hacen antes de las 14:00.

### Facturas
La factura se emite el mismo día de la compra y se envía por correo. Si necesitas factura con CIF, se emite en 24 h con datos fiscales completos.

### Devoluciones
Hay 30 días naturales para devolver. El producto debe estar sin usar. El envío de devolución corre a cargo del cliente salvo producto defectuoso.

### Problemas de entrega
Si el paquete llega dañado, haz un ticket con foto en menos de 48 h. Si no llega en 7 días, contacta con soporte: el pedido se re-envía gratis."""
_vec = __import__("sklearn.feature_extraction.text", fromlist=["TfidfVectorizer"]).TfidfVectorizer(ngram_range=(1, 2))

def chunking_solape(doc, solape=0.15):
    partes = re.split(r"\n(?=### )", doc)
    chunks = []
    for p in partes:
        lineas = p.strip().split("\n")
        titulo = lineas[0].replace("### ", "")
        cuerpo = "\n".join(lineas[1:])
        palabras = cuerpo.split()
        corte = int(len(palabras) * (1 - solape))
        chunks.append({"id": f"{titulo} [a]", "texto": f"{titulo}: {' '.join(palabras[:corte])}"})
        chunks.append({"id": f"{titulo} [b]", "texto": f"{titulo}: {' '.join(palabras[corte - 2:])}"})
    return chunks

chunks_s = chunking_solape(GUIA)
print(f"{len(chunks_s)} chunks con solape")
_mat_chunks = _vec.fit_transform([c["texto"] for c in chunks_s]).toarray()
chunks = chunks_s


def _normalizar(v):
    return v / (np.linalg.norm(v) + 1e-9)

def buscar(top_k=2, consulta: str = "", **ctx):
    q = _normalizar(encodar(consulta))
    sims = _mat_chunks @ _normalizar(q)
    idx = np.argsort(sims)[::-1][:top_k]
    return [{"id": chunks[i]["id"], "texto": chunks[i]["texto"], "score": float(sims[i])} for i in idx]

def encodar(texto: str):
    return _vec.transform([texto]).toarray()[0]


for r in buscar(top_k=3, consulta="cuánto tarda el envío"):
    print(f"[{r['id']}] score={r['score']:.2f}")

**Lectura:** el solape duplica el índice pero ninguna frase cruzada queda fuera. En secciones cortas la ganancia es pequeña; en documentos largos es imprescindible.

## E2 · Documento adicional y límites del router

In [ ]:
DOC2 = """### Devoluciones extendidas
Para productos electrónicos hay 60 días. La garantía es de 2 años
según la ley de consumidores y no consume los 30 días de devolución.

### Contacto
El soporte responde en 24 h en días laborables.
El teléfono está disponible de 9 a 14 h."""

def chunking_generico(doc):
    partes = [p for p in re.split(r"\n(?=### )", doc) if p.strip() and "### " in p]
    out = []
    for p in partes:
        lineas = p.strip().split("\n")
        titulo = lineas[0].replace("### ", "")
        cuerpo = " ".join(l.strip() for l in lineas[1:])
        out.append({"id": titulo, "texto": f"{titulo}: {cuerpo}"})
    return out

chunks_e2 = chunking_generico(GUIA.split("\n\n", 1)[1] + "\n\n" + DOC2.strip())
_mat_chunks = _vec.fit_transform([c["texto"] for c in chunks_e2]).toarray()
chunks = chunks_e2
for r in buscar(top_k=2, consulta="¿cuánto dura la garantía?"):
    print(f"[{r['id']}] score={r['score']:.2f}")

**Lectura:** la consulta 'garantía' mezcla secciones y el router por título del top-1 empieza a fallar. Mitigaciones: no decidir con un solo chunk, usar score mínimo o resumir los 2 top.

## E3 · Embeddings reales (Colab o local con HF)

In [ ]:
# Ejecutar donde haya internet: pip install -U sentence-transformers
# from sentence_transformers import SentenceTransformer
# modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
#
# _emb_chunks = np.vstack([modelo.encode(c["texto"]) for c in chunks])
#
# def buscar_semantica(consulta, top_k=2):
#     q = modelo.encode([consulta])[0]
#     sims = (_emb_chunks @ q) / (np.linalg.norm(_emb_chunks, axis=1) * np.linalg.norm(q) + 1e-9)
#     idx = np.argsort(sims)[::-1][:top_k]
#     return [{"id": chunks[i]["id"], "texto": chunks[i]["texto"], "score": float(sims[i])} for i in idx]
#
# for r in buscar_semantica("¿reenvían gratis si se pierde el paquete?"):
#     print(f"[{r['id']}] {r['score']:.2f}  {r['texto'][:60]}")
print("esqueleto listo (requiere internet/GPU)")

**Esperado:** con embeddings semánticos, sinónimos y parafrasis ('reenvían gratis' vs 'se re-envía gratis') recuperan el chunk mejor que TF-IDF, cuyo matching literal de n-gramas falla.

## E4 · Herramienta responder(tema)

In [ ]:
def responder(tema: str) -> dict:
    """Herramienta pedida en el ejercicio: el agente 'cierra' con esta."""
    return {"cerrado": True, "tema": tema}

# problema: 'responder' como herramienta abre el bucle responder→responder
# y el agente puede terminar sin respuesta visible.
# cierre correcto: la respuesta final del modelo (stop_reason != 'tool_use').
print("conclusión: no cerrar con una herramienta falsa")

**Respuesta:** duplica el mecanismo de cierre y abre llamadas infinitas. El cierre correcto es la respuesta final del asistente.

## E5 · Router determinista para 'cambiar dirección'

```python
def cambiar_direccion(mensaje):
    if "cambiar" in mensaje and "dirección" in mensaje:
        return "El cambio debe hacerlo en la app: Perfil > Pedidos > Editar."
    return "No entiendo la solicitud de dirección."
```
**Por qué es mejor que un agente:** son 2 casos conocidos; el router es barato, determinista y testable. El agente añade latencia, coste e imprecisión.